**QCNN:**

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self, cnn_features):
        super().__init__()
        self.features = cnn_features

        with torch.no_grad():
            dummy = torch.zeros(1, 3, 200, 200).to(next(self.features.parameters()).device)
            out = self.features(dummy)
            feature_dim = out.view(1, -1).size(1)

        self.fc = nn.Linear(feature_dim, 4)  # quantum features

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [ ]:
# class HybridQCNN(nn.Module):
#     def __init__(self, qnn):
#         super().__init__()
#         self.qnn = qnn
#         self.fc = nn.Linear(16, 1)  # 🔑 2 observables → 1 logit

#     def forward(self, x):
#         x = self.qnn(x)
#         return self.fc(x)
class HybridQCNN(nn.Module):
    def __init__(self, qnn):
        super().__init__()
        self.qnn = qnn
        self.fc1 = nn.Linear(4, 16)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.qnn(x)
        x = self.act(self.fc1(x))
        return self.fc2(x)



In [ ]:
feature_model = FeatureExtractor(model).to(device)
feature_model.eval()

X_train, y_train = [], []

with torch.no_grad():
    for x, y in train_loader:
        x = x.to(device)
        features = feature_model(x).cpu().numpy()
        X_train.extend(features)
        y_train.extend(y.numpy())


In [ ]:
# TRAIN FEATURE NORMALIZATION (PER-FEATURE)
X_train = np.array(X_train)
y_train = np.array(y_train)

# compute per-feature min/max on TRAIN ONLY
mins = X_train.min(axis=0)
maxs = X_train.max(axis=0)

# normalize each feature independently
X_train = (X_train - mins) / (maxs - mins + 1e-8)

# map to quantum angle range
X_train = X_train * np.pi   # [0, pi]

# BALANCE DATA FOR QCNN
from sklearn.utils import resample

# separate classes
pos = X_train[y_train == 1]
neg = X_train[y_train == 0]

# take equal samples
n = min(len(pos), len(neg))

X_bal = np.vstack([pos[:n], neg[:n]])
y_bal = np.hstack([np.ones(n), np.zeros(n)])

# shuffle
idx = np.random.permutation(len(X_bal))
X_bal = X_bal[idx]
y_bal = y_bal[idx]

# limit QCNN samples
X_train_q = X_bal[:200]
y_train_q = y_bal[:200]

print("QCNN class balance:", np.mean(y_train_q))  # should be ~0.5


QCNN class balance: 0.505


In [ ]:
num_qubits = 4

feature_map = ZFeatureMap(num_qubits)
ansatz = TwoLocal(
    num_qubits,
    rotation_blocks=["ry", "rz"],
    entanglement_blocks="cx",
    reps=2,     # increase depth
    entanglement="full" # full or linear
)

qc = QuantumCircuit(num_qubits)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)
# qc.draw("mpl")

In [ ]:
# backend = AerSimulator()
# optimizer = COBYLA(maxiter=100)
# estimator = Estimator()
from qiskit.primitives import Estimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_aer.primitives import Estimator as AerEstimator

# estimator = AerEstimator(run_options={"shots": 1024})
from qiskit.primitives import Estimator
estimator = Estimator()   # analytic, NO shots

observables = [
    SparsePauliOp.from_list([("ZIII", 1.0)]),
    SparsePauliOp.from_list([("IZII", 1.0)]),
    SparsePauliOp.from_list([("IIZI", 1.0)]),
    SparsePauliOp.from_list([("IIIZ", 1.0)]),
]

qnn = EstimatorQNN(
    circuit=qc,
    observables=observables,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    estimator=estimator,
)

base_qnn = TorchConnector(qnn).to(device)
model_qcnn = HybridQCNN(base_qnn).to(device)

# vqc.fit(X_train, y_train)


In [ ]:
optimizer_q = optim.Adam(model_qcnn.parameters(), lr=0.01) # If unstable: lr = 0.005
criterion_q = nn.BCEWithLogitsLoss()
X_q = torch.tensor(X_train_q, dtype=torch.float32).to(device)
y_q = torch.tensor(y_train_q, dtype=torch.float32).view(-1, 1).to(device)

batch_size = 16

for epoch in range(50):
    perm = torch.randperm(len(X_q))
    total_loss = 0

    for i in range(0, len(X_q), batch_size):
        idx = perm[i:i+batch_size]
        xb = X_q[idx]
        yb = y_q[idx]

        optimizer_q.zero_grad()
        outputs = model_qcnn(xb)
        loss = criterion_q(outputs, yb)
        loss.backward()
        optimizer_q.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 8.5815
Epoch 1, Loss: 7.1466
Epoch 2, Loss: 5.9478
Epoch 3, Loss: 5.2447
Epoch 4, Loss: 4.4466
Epoch 5, Loss: 4.0579
Epoch 6, Loss: 3.5482
Epoch 7, Loss: 3.2544
Epoch 8, Loss: 2.9393
Epoch 9, Loss: 2.8327
Epoch 10, Loss: 2.8021
Epoch 11, Loss: 2.5159
Epoch 12, Loss: 2.4739
Epoch 13, Loss: 2.5624
Epoch 14, Loss: 2.4608
Epoch 15, Loss: 2.1690
Epoch 16, Loss: 2.1033
Epoch 17, Loss: 2.0674
Epoch 18, Loss: 2.0211
Epoch 19, Loss: 2.1617
Epoch 20, Loss: 1.9436
Epoch 21, Loss: 1.9505
Epoch 22, Loss: 1.9443
Epoch 23, Loss: 1.9546
Epoch 24, Loss: 1.9585
Epoch 25, Loss: 1.9323
Epoch 26, Loss: 1.9430
Epoch 27, Loss: 1.8150
Epoch 28, Loss: 1.8109
Epoch 29, Loss: 1.7553
Epoch 30, Loss: 1.7535
Epoch 31, Loss: 1.7504
Epoch 32, Loss: 1.7864
Epoch 33, Loss: 2.0993
Epoch 34, Loss: 1.7159
Epoch 35, Loss: 1.8527
Epoch 36, Loss: 1.7786
Epoch 37, Loss: 1.5916
Epoch 38, Loss: 1.5654
Epoch 39, Loss: 1.5461
Epoch 40, Loss: 1.5112
Epoch 41, Loss: 1.5988
Epoch 42, Loss: 1.5940
Epoch 43, Loss: 1.442

In [ ]:
X_test, y_test = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        feats = feature_model(x).cpu().numpy()
        X_test.extend(feats)
        y_test.extend(y.numpy())

X_test = np.array(X_test)
y_test = np.array(y_test)

# NORMALIZE TEST USING TRAIN STATS
X_test = (X_test - mins) / (maxs - mins + 1e-8)
X_test = X_test * np.pi


In [ ]:
# Convert to torch
X_t = torch.tensor(X_test, dtype=torch.float32).to(device)

model_qcnn.eval()

with torch.no_grad():
    outputs = model_qcnn(X_t)
    probs = torch.sigmoid(outputs).cpu().numpy()
    preds = (probs > 0.5).astype(int)

acc = accuracy_score(y_test, preds)
print("QCNN Accuracy:", acc)


QCNN Accuracy: 0.6816666666666666
